In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/lopezjatjat10@gmail.com/sportsbar-etl-proj/1_setup/utilities


In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'customers', 'Data Source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sporsbar-proj-lopez/{data_source}/*.csv'

print(base_path)

In [0]:
df = (spark.read.format('csv')
      .option('header', True)
      .option('inferSchema', True)
      .load(base_path)
      .withColumn('read_timestamp', F.current_timestamp())
      .select('*', '_metadata.file_name', '_metadata.file_size')
)


display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
(df.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}'))